In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [9]:
class AlexNetBackbone(nn.Module):
    def __init__(self, in_channels=3, use_last_pool=False):
        super().__init__()
        self.use_last_pool = use_last_pool
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 96, kernel_size=11, stride=4, padding=0),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.pool5 = nn.MaxPool2d(kernel_size=3, stride=2)
        self._init_weights()

    def _init_weights(self):
        # Ініціалізація згідно з Krizhevsky et al., 2012
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        if self.use_last_pool:
            x = self.pool5(x)
        return x


In [10]:
class DetectionHead(nn.Module):
    def __init__(self, in_channels, num_classes, num_anchors, dropout_p=0.5):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 512, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout_p)
        self.cls_head = nn.Conv2d(512, num_anchors * num_classes, kernel_size=1)
        self.box_head = nn.Conv2d(512, num_anchors * 4, kernel_size=1)

        # Ініціалізація
        for m in [self.conv, self.cls_head, self.box_head]:
            nn.init.normal_(m.weight, 0, 0.01)
            nn.init.constant_(m.bias, 0)

    def forward(self, feat):
        x = self.relu(self.conv(feat))
        x = self.dropout(x)
        cls = self.cls_head(x)
        box = self.box_head(x)
        return cls, box

In [11]:
class AlexNetDetector(nn.Module):
    def __init__(self, num_classes=21, num_anchors=9):
        super().__init__()
        self.backbone = AlexNetBackbone()
        self.head = DetectionHead(in_channels=256, num_classes=num_classes, num_anchors=num_anchors)

    def forward(self, x):
        feat = self.backbone(x)
        cls, box = self.head(feat)
        return cls, box

In [12]:
model = AlexNetDetector(num_classes=21)
x = torch.randn(2, 3, 224, 224)
cls, box = model(x)
print(f"Вихід класифікатора: {cls.shape}")
print(f"Вихід регресора боксів: {box.shape}")

Вихід класифікатора: torch.Size([2, 189, 12, 12])
Вихід регресора боксів: torch.Size([2, 36, 12, 12])


In [13]:
hyperparams = {
    "batch_size": 128,
    "learning_rate": 0.01,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "dropout": 0.5,
}
print("\n  Гіперпараметри (з AlexNet paper):")
for k, v in hyperparams.items():
    print(f"{k:>15}: {v}")


  Гіперпараметри (з AlexNet paper):
     batch_size: 128
  learning_rate: 0.01
       momentum: 0.9
   weight_decay: 0.0005
        dropout: 0.5


In [14]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
print("\n✅ Оптимізатор створено (SGD).")


✅ Оптимізатор створено (SGD).


In [15]:
# -----------------------
# 1️⃣  Створюємо модель
# -----------------------
model = AlexNetDetector(num_classes=21, num_anchors=9)

# Dropout уже "вшитий" у модель
print(model.head.dropout)  # nn.Dropout(p=0.5)

# -----------------------
# 2️⃣  Створюємо оптимізатор
# -----------------------
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,          # learning_rate
    momentum=0.9,     # momentum
    weight_decay=5e-4 # L2 регуляризація
)

# -----------------------
# 3️⃣  DataLoader із batch_size=128
# -----------------------
from torch.utils.data import DataLoader, TensorDataset

# (приклад на фейкових даних)
X = torch.randn(512, 3, 224, 224)
y = torch.randint(0, 21, (512,))
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

# -----------------------
# 4️⃣  Цикл тренування (спрощений)
# -----------------------
for epoch in range(2):
    for xb, yb in loader:
        cls_pred, box_pred = model(xb)
        loss = cls_pred.mean()  # (для прикладу)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}: done ✅")


Dropout(p=0.5, inplace=False)
Epoch 1: done ✅
Epoch 2: done ✅
